In [1]:
%pip install faiss-cpu sentence-transformers requests


   ---------------------------------------- 0.0/16.2 MB ? eta -:--:--
   ---------------------------------------- 0.0/16.2 MB ? eta -:--:--
   ---------------------------------------- 0.0/16.2 MB ? eta -:--:--
   ---------------------------------------- 0.0/16.2 MB ? eta -:--:--
   ---------------------------------------- 0.0/16.2 MB ? eta -:--:--
   ---------------------------------------- 0.0/16.2 MB ? eta -:--:--
    --------------------------------------- 0.3/16.2 MB ? eta -:--:--
    --------------------------------------- 0.3/16.2 MB ? eta -:--:--
   - -------------------------------------- 0.5/16.2 MB 720.4 kB/s eta 0:00:22
   - -------------------------------------- 0.8/16.2 MB 753.6 kB/s eta 0:00:21
   - -------------------------------------- 0.8/16.2 MB 753.6 kB/s eta 0:00:21
   -- ------------------------------------- 1.0/16.2 MB 805.0 kB/s eta 0:00:19
   --- ------------------------------------ 1.3/16.2 MB 879.6 kB/s eta 0:00:17
   --- ------------------------------------ 

In [2]:
# imports
import os
import faiss
import numpy as np
import requests
from sentence_transformers import SentenceTransformer

In [3]:
# load all txt files from the docs folder
docs_folder = "docs"

documents = []

for filename in sorted(os.listdir(docs_folder)):
    if filename.endswith(".txt"):
        filepath = os.path.join(docs_folder, filename)
        with open(filepath, "r", encoding="utf-8") as f:
            text = f.read()
        documents.append({"filename": filename, "text": text})

print(f"Loaded {len(documents)} documents")

# chunk each document by paragraph, grouping up to ~120 words per chunk
def chunk_text(text, max_words=120):
    paragraphs = [p.strip() for p in text.split("\n\n") if p.strip()]
    chunks = []
    current_chunk = ""

    for para in paragraphs:
        if len((current_chunk + " " + para).split()) > max_words and current_chunk:
            chunks.append(current_chunk.strip())
            current_chunk = para
        else:
            current_chunk = (current_chunk + " " + para).strip()

    if current_chunk:
        chunks.append(current_chunk.strip())

    return chunks

all_chunks = []

for doc in documents:
    doc_chunks = chunk_text(doc["text"], max_words=120)
    for i, chunk in enumerate(doc_chunks):
        all_chunks.append({
            "text": chunk,
            "source": doc["filename"],
            "chunk_id": f"{doc['filename']}_{i}"
        })

print(f"Total chunks created: {len(all_chunks)}")

Loaded 10 documents
Total chunks created: 35


In [4]:
# load the same embedding model as Task 3
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

# extract chunk texts and embed them
chunk_texts = [chunk["text"] for chunk in all_chunks]
chunk_embeddings = embedding_model.encode(chunk_texts, show_progress_bar=True)

# faiss needs float32 numpy arrays
chunk_embeddings = np.array(chunk_embeddings).astype("float32")

print(f"Generated {chunk_embeddings.shape[0]} embeddings of dimension {chunk_embeddings.shape[1]}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/2 [00:00<?, ?it/s]

Generated 35 embeddings of dimension 384


In [5]:
# create a FAISS index using L2 (Euclidean) distance
dimension = chunk_embeddings.shape[1]  # 384
index = faiss.IndexFlatL2(dimension)

# add all chunk embeddings to the index
index.add(chunk_embeddings)

print(f"FAISS index built with {index.ntotal} vectors")

FAISS index built with 35 vectors


In [6]:
def retrieve(query, top_k=3):
    # embed the query using the same model
    query_embedding = embedding_model.encode([query]).astype("float32")

    # search the FAISS index for the top_k nearest chunks
    distances, indices = index.search(query_embedding, top_k)

    # gather the matching chunks
    results = []
    for idx, dist in zip(indices[0], distances[0]):
        chunk = all_chunks[idx]
        results.append({
            "text": chunk["text"],
            "source": chunk["source"],
            "distance": float(dist)
        })

    return results

# quick test
test_results = retrieve("What is LoRA used for?")
for r in test_results:
    print(f"Source: {r['source']} | Distance: {r['distance']:.4f}")
    print(r["text"][:150] + "...")
    print()

Source: 06_lora.txt | Distance: 0.8949
LoRA (Low-Rank Adaptation) LoRA, which stands for Low-Rank Adaptation, is a technique used to fine-tune large pretrained models efficiently, without u...

Source: 06_lora.txt | Distance: 0.8950
LoRA is especially popular for adapting large language models to narrow tasks, such as adjusting a model's tone or style, teaching it a specific forma...

Source: 06_lora.txt | Distance: 1.1462
After training, the small LoRA adapter can be saved separately from the base model, often as a file only a few megabytes in size. This adapter can lat...



In [7]:
def generate_answer(query, top_k=3):
    # retrieve relevant chunks
    results = retrieve(query, top_k=top_k)

    # build context string from retrieved chunks
    context = "\n\n".join([f"[Source: {r['source']}]\n{r['text']}" for r in results])

    # build a prompt that forces the model to answer ONLY from the provided context
    prompt = f"""Answer the question using ONLY the information in the context below.
If the context does not contain enough information to answer, say "I cannot answer this based on the provided documents."
Do not use any outside knowledge.

Context:
{context}

Question: {query}

Answer:"""

    # call Ollama's local API
    response = requests.post(
        "http://localhost:11434/api/generate",
        json={
            "model": "qwen2.5:3b",
            "prompt": prompt,
            "stream": False
        }
    )

    answer = response.json()["response"]

    return answer, results

# quick test
answer, sources = generate_answer("What is LoRA used for?")
print("ANSWER:")
print(answer)
print("\nSOURCES USED:")
for r in sources:
    print(f"- {r['source']}")

ANSWER:
LoRA is used for adapting large language models to narrow tasks such as adjusting a model's tone or style, teaching it a specific format, or specializing it for a particular domain. It does this by fine-tuning only the small, trainable low-rank matrices alongside certain layers of the model, typically the attention layers, while freezing the original pretrained weights entirely. This technique makes fine-tuning large models more practical and efficient on limited hardware like a single consumer GPU.

SOURCES USED:
- 06_lora.txt
- 06_lora.txt
- 06_lora.txt


In [8]:
for q in [
    "How does gradient descent work?",
    "What's the difference between precision and recall?",
    "How do transformers use self-attention?",
]:
    answer, sources = generate_answer(q)
    print(f"Q: {q}")
    print(f"A: {answer}")
    print(f"Sources: {[s['source'] for s in sources]}")
    print("-" * 60)

Q: How does gradient descent work?
A: Gradient descent works by adjusting the parameters of a machine learning model in order to minimize a loss function. The core idea is to compute the gradient (the slope) of the loss function with respect to each parameter, indicating the direction in which the loss would increase the fastest. By moving the parameters in the opposite direction of this gradient, the algorithm gradually reduces the loss, step by step.

There are several variations of gradient descent:

1. Batch Gradient Descent: This method computes the gradient using the entire training dataset before making a single update.
2. Stochastic Gradient Descent (SGD): In contrast to batch gradient descent, which uses all data points at once, SGD updates the parameters using just one training example at a time.
3. Mini-batch Gradient Descent: This is the most commonly used approach in practice. Instead of using all or one data point, it computes the gradient over small batches (mini-batches

In [ ]:
answer, sources = generate_answer("What is the capital of France?")
print(f"Q: What is the capital of France?")
print(f"A: {answer}")
print(f"Sources: {[s['source'] for s in sources]}")